# MDVR_KCL Speech Model Training

Dataset: MDVR_KCL (Multi-Modality Dysarthria Voice Research - King's College London)
- ReadText & SpontaneousDialogue tasks
- HC (healthy control) vs PD (Parkinson's disease) classification
- .wav audio recordings

Pattern: Extract acoustic features → XGBoost classifier → SHAP explanations
Compatible with Oxford and UCI speech sub-models for feature space alignment

## 1. Import Required Libraries

In [39]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
 
from sklearn.model_selection import (GroupShuffleSplit,StratifiedGroupKFold)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.frozen import FrozenEstimator
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix,
)

from preprocess import (
    process_mdvr_dataset,
    features_to_dataframe
)

In [28]:
RANDOM_STATE = 42

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

# Update with your dataset path
MDVR_DATASET_ROOT = r"datasets\speech\MDVR_KCL\26_29_09_2017_KCL"

data = process_mdvr_dataset(MDVR_DATASET_ROOT)
df = features_to_dataframe(data)
print(df.shape)

df.head()

In [3]:
EXCLUDE = ["record_id", "label", "subject_id", "file", "task","class_type"]

feature_cols = [c for c in df.columns if c not in EXCLUDE]
print("Features:", len(feature_cols))

Features: 592


In [4]:
file_path = ARTIFACT_DIR/"data.csv"

# 4. Save the DataFrame to the folder
df.to_csv(file_path, index=False)
print(f"File successfully saved to: {file_path}")

File successfully saved to: artifacts\data.csv


In [5]:
df = pd.read_csv(file_path)

In [26]:
META_COLS = {"record_id", "label", "subject_id", "file", "task", "class_type"}
feature_cols = [c for c in df.columns if c not in META_COLS]
 
X = df[feature_cols].values.astype(np.float64)
y = df["label"].values.astype(int)
groups = df["subject_id"].values
 
print(f"Rows: {len(df)}, Features: {len(feature_cols)}, Subjects: {len(set(groups))}")
print(f"Class balance: {np.bincount(y)}  (PD ratio: {np.mean(y):.3f})")
print()

Rows: 73, Features: 592, Subjects: 38
Class balance: [42 31]  (PD ratio: 0.425)



In [29]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
 
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
groups_train = groups[train_idx]
 
s_train, s_test = set(groups[train_idx]), set(groups[test_idx])
assert s_train.isdisjoint(s_test), "LEAKAGE: subject in both train and test"
 
print(f"Train: {len(train_idx)} rows / {len(s_train)} subjects")
print(f"Test:  {len(test_idx)} rows / {len(s_test)} subjects")
print()

Train: 57 rows / 30 subjects
Test:  16 rows / 8 subjects



In [48]:
K_CANDIDATES = [20, 30, 40, 50, 60, 80]
gkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)
 
print("Sweeping k_features via grouped 5-fold CV:")
best_k, best_k_auc = None, -1.0
 
for k in K_CANDIDATES:
    fold_aucs = []
    for tr_i, va_i in gkf.split(X_train, y_train, groups=groups_train):
        sel = SelectKBest(mutual_info_classif,k=k)
        X_tr_k = sel.fit_transform(X_train[tr_i], y_train[tr_i])
        X_va_k = sel.transform(X_train[va_i])
 
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=RANDOM_STATE)),
        ])
        # Fit on DataFrame to avoid the "fitted without feature names" warning
        sel_names = [feature_cols[i] for i in sel.get_support(indices=True)]
        pipe.fit(
            pd.DataFrame(X_tr_k, columns=sel_names),
            y_train[tr_i],
        )
        probs = pipe.predict_proba(
            pd.DataFrame(X_va_k, columns=sel_names)
        )[:, 1]
        fold_aucs.append(roc_auc_score(y_train[va_i], probs))
 
    mean_auc = float(np.mean(fold_aucs))
    print(f"  k={k:4d}  grouped CV AUC = {mean_auc:.4f} (+/- {np.std(fold_aucs):.4f})")
    if mean_auc > best_k_auc:
        best_k, best_k_auc = k, mean_auc
 
print(f"\nBest k_features by grouped CV: {best_k} (AUC={best_k_auc:.4f})")
print(best_k)
print(best_k_auc)

Sweeping k_features via grouped 5-fold CV:
  k=  20  grouped CV AUC = 0.7616 (+/- 0.1945)
  k=  30  grouped CV AUC = 0.7482 (+/- 0.2012)
  k=  40  grouped CV AUC = 0.8063 (+/- 0.1715)
  k=  50  grouped CV AUC = 0.8313 (+/- 0.1752)
  k=  60  grouped CV AUC = 0.8500 (+/- 0.1868)
  k=  80  grouped CV AUC = 0.7969 (+/- 0.1899)

Best k_features by grouped CV: 60 (AUC=0.8500)
60
0.85


In [49]:
selector = selector = SelectKBest(mutual_info_classif,k=best_k)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)
 
selected_idx = selector.get_support(indices=True)
selected_feature_names = [feature_cols[i] for i in selected_idx]

print("\nTop Selected Features:")
for i, f in enumerate(selected_feature_names, 1):
    print(f"{i:02d}. {f}")

feature_map = {name: name for name in selected_feature_names}


Top Selected Features:
01. MDVP_Fo_Hz_
02. pitch_median
03. pitch_q25
04. pitch_q75
05. Shimmer_APQ3
06. Shimmer_APQ5
07. NHR
08. HNR
09. zcr_q25
10. rms_std
11. spectral_centroid_mean
12. spectral_bandwidth_mean
13. spectral_rolloff_range
14. spectral_rolloff_skew
15. spectral_flatness_max
16. spectral_flatness_range
17. spectral_contrast_1_max
18. spectral_contrast_2_std
19. spectral_contrast_3_min
20. spectral_contrast_4_range
21. spectral_contrast_4_skew
22. spectral_contrast_5_std
23. spectral_contrast_5_max
24. delta_mfcc_1_std
25. delta2_mfcc_1_std
26. delta2_mfcc_1_min
27. delta2_mfcc_1_range
28. mfcc_2_mean
29. delta_mfcc_2_min
30. delta2_mfcc_2_q75
31. mfcc_3_std
32. mfcc_3_q25
33. delta_mfcc_3_kurtosis
34. delta2_mfcc_3_min
35. delta2_mfcc_3_q75
36. delta2_mfcc_3_skew
37. mfcc_4_q75
38. mfcc_4_range
39. delta2_mfcc_4_min
40. mfcc_5_std
41. delta_mfcc_5_q25
42. delta_mfcc_5_range
43. delta_mfcc_6_min
44. mfcc_7_std
45. mfcc_7_median
46. mfcc_7_q25
47. mfcc_7_kurtosis
48. del

In [41]:
gss_cal = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=RANDOM_STATE
)

tr_idx, cal_idx = next(
    gss_cal.split(
        X_train_sel,
        y_train,
        groups_train
    )
)

X_train_final = X_train_sel[tr_idx]
y_train_final = y_train[tr_idx]

X_cal = X_train_sel[cal_idx]
y_cal = y_train[cal_idx]

X_train_final_df = pd.DataFrame(
    X_train_final,
    columns=selected_feature_names
)

X_cal_df = pd.DataFrame(
    X_cal,
    columns=selected_feature_names
)

In [42]:
base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=RANDOM_STATE)),
])
# Pass DataFrame so imputer is fitted with feature names (avoids warning at inference)
X_train_df = pd.DataFrame(X_train_sel, columns=selected_feature_names)
base_model.fit(X_train_final_df, y_train_final)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

In [43]:
cal_model = CalibratedClassifierCV(
    estimator=FrozenEstimator(base_model),
    method="sigmoid",
)
cal_model.fit(X_cal_df,y_cal)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",FrozenEstimat..._state=42))]))
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""m

In [44]:
X_test_df = pd.DataFrame(X_test_sel, columns=selected_feature_names)
test_probs = cal_model.predict_proba(X_test_df)[:, 1]
preds = (test_probs >= 0.5).astype(int)
val_auc = roc_auc_score(y_test, test_probs)
 
print()
print("========== TEST RESULTS @ threshold=0.5 ==========")
print(f"ROC-AUC  : {val_auc:.4f}")
print(f"Accuracy : {accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score : {f1_score(y_test, preds, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, preds))


========== TEST RESULTS @ threshold=0.5 ==========
ROC-AUC  : 0.8833
Accuracy : 0.8750
Precision: 0.9000
Recall   : 0.9000
F1 Score : 0.9000
Confusion matrix:
 [[5 1]
 [1 9]]


In [47]:
N_BOOTSTRAP = 50
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_models = []
n_train = X_train_final.shape[0]
 
print(f"\nTraining {N_BOOTSTRAP} bootstrap models...")
for i in range(N_BOOTSTRAP):
    boot_idx = rng.integers(0, n_train, size=n_train)
    X_boot = pd.DataFrame(X_train_final[boot_idx], columns=selected_feature_names)
    y_boot = y_train_final[boot_idx]
 
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=RANDOM_STATE + i)),
    ])
    pipe.fit(X_boot, y_boot)
    bootstrap_models.append(pipe)
 
print(f"Bootstrap models trained. MC range check on sample:")
sample_row = pd.DataFrame(X_train_final[[0]], columns=selected_feature_names)
mc_check = [float(m.predict_proba(sample_row)[0, 1]) for m in bootstrap_models]
print( f"MC mean = {np.mean(mc_check):.3f}")
print(f"MC std  = {np.std(mc_check):.3f}")
print(f"MC range: [{min(mc_check):.3f}, {max(mc_check):.3f}]")


Training 50 bootstrap models...
Bootstrap models trained. MC range check on sample:
MC mean = 0.098
MC std  = 0.095
MC range: [0.015, 0.437]


In [50]:
N_BACKGROUND = min(20, X_train_final.shape[0])

bg_idx = rng.choice(
    X_train_final.shape[0],
    size=N_BACKGROUND,
    replace=False
)

shap_background = X_train_final[bg_idx]

print(
    f"\nSHAP background sample: "
    f"{shap_background.shape} "
    f"(subset of training set)"
)


SHAP background sample: (20, 60) (subset of training set)


In [ ]:
decision_threshold = 0.5 
 
joblib.dump(base_model,              ARTIFACT_DIR / "speech_mdvr_best.pkl")
joblib.dump(cal_model,               ARTIFACT_DIR / "speech_mdvr_best_calibrated.pkl")
joblib.dump(feature_map,             ARTIFACT_DIR / "feature_map.pkl")
joblib.dump(selected_feature_names,  ARTIFACT_DIR / "selected_feature_names.pkl")
joblib.dump(decision_threshold,      ARTIFACT_DIR / "decision_threshold.pkl")
joblib.dump(bootstrap_models,        ARTIFACT_DIR / "bootstrap_models.pkl")
joblib.dump(shap_background,         ARTIFACT_DIR / "shap_background.pkl")
joblib.dump(val_auc,                 ARTIFACT_DIR / "validation_auc.pkl")
 
metadata = {
    "model_type":           "SVC_RBF_Pipeline",
    "k_features":           best_k,
    "n_train_rows":         int(len(train_idx)),
    "n_test_rows":          int(len(test_idx)),
    "n_subjects_train":     int(len(s_train)),
    "n_subjects_test":      int(len(s_test)),
    "n_bootstrap_models":   N_BOOTSTRAP,
    "n_shap_background":    N_BACKGROUND,
    "decision_threshold":   decision_threshold,
    "auc":                  round(float(val_auc), 4),
    "accuracy":             round(float(accuracy_score(y_test, preds)), 4),
    "precision":            round(float(precision_score(y_test, preds, zero_division=0)), 4),
    "recall":               round(float(recall_score(y_test, preds, zero_division=0)), 4),
    "f1":                   round(float(f1_score(y_test, preds, zero_division=0)), 4),
}
with open(ARTIFACT_DIR / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)